# 05_Material_Control_and_Logistics.ipynb

# 0. SETUP

In [1]:
# ============================================================
# 0. SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# Rutas del proyecto (ajusta si es necesario)
BASE_DIR = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")
OUTPUT_DIR = BASE_DIR / "05_Outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Carga de datos previos
# (Usamos el Activity Master más completo disponible)
df_master = pd.read_excel(BASE_DIR / "03_Outputs" / "Activity_Master.xlsx")
df_tracking = pd.read_excel(BASE_DIR / "04_Outputs" / "Project_Tracking_with_Alarms.xlsx")

print("=" * 70)
print("SETUP COMPLETADO")
print("=" * 70)
print(f"Activity Master     : {len(df_master)} actividades")
print(f"Project Tracking    : {len(df_tracking)} periodos")
print(f"Columnas Master     : {list(df_master.columns)}")
print(f"Output directory    : {OUTPUT_DIR}")

# 1. Diccionario de reglas + Etiquetado de materiales críticos

In [2]:
# ============================================================
# 1. DICCIONARIO DE REGLAS DE CLASIFICACIÓN (v1.2) – Ajustado al dataset real
# ============================================================

MATERIAL_RULES = {
    "tower": {
        "keywords": ["tower", "towers", "toren", "torens", "mast", "masten"],
        "material_type": "Tower",
        "lead_time_days": 240,
        "buffer_days": 21,
        "criticality": "Critical",
        "notes": "Long-lead. Transporte especial. Secuencia de montaje estricta."
    },
    "nacelle": {
        "keywords": [
            "nacelle", "gondel", "gondels", "machine house", "hub",
            "erection machines", "erection machine", "machines"   # ← clave del dataset
        ],
        "material_type": "Nacelle",
        "lead_time_days": 300,
        "buffer_days": 28,
        "criticality": "Critical",
        "notes": "Incluye Nacelle + Hub + Blades (paquete 'Erection machines'). Componente más crítico."
    },
    "blade": {
        "keywords": ["blade", "blades", "pala", "palas", "rotor blade", "wieken", "wiek"],
        "material_type": "Blade",
        "lead_time_days": 210,
        "buffer_days": 21,
        "criticality": "Critical",
        "notes": "En este dataset va integrado en 'Erection machines'."
    },
    "transformer": {
        "keywords": ["transformer", "transformers", "transformator", "transformatoren", "trafo"],
        "material_type": "Transformer",
        "lead_time_days": 270,
        "buffer_days": 30,
        "criticality": "High",
        "notes": "Especiales. Largos plazos de fabricación."
    },
    "cable": {
        "keywords": [
            "cable", "cables", "power cable", "medium voltage", "high voltage",
            "kabel", "kabels", "mv cable", "hv cable",
            "electrical connection"          # ← añadido
        ],
        "material_type": "Cable",
        "lead_time_days": 120,
        "buffer_days": 14,
        "criticality": "Medium-High",
        "notes": "Incluye conexión eléctrica a turbinas."
    },
    "foundation": {
        "keywords": ["foundation", "foundations", "fundament", "funderingen",
                     "pile", "piles", "paal", "palen", "footing"],
        "material_type": "Foundation",
        "lead_time_days": 75,
        "buffer_days": 14,
        "criticality": "Medium",
        "notes": "Obra civil. Menos long-lead que los componentes de turbina."
    },
    "crane": {
        "keywords": ["crane", "cranes", "kraan", "kranen", "main crane", "crawler"],
        "material_type": "Crane",
        "lead_time_days": 50,
        "buffer_days": 14,
        "criticality": "High",
        "notes": "Recurso crítico de montaje."
    }
}

# Orden de prioridad (importante: nacelle antes que tower por si hubiera solapamiento)
PRIORITY_ORDER = ["nacelle", "blade", "tower", "transformer", "crane", "cable", "foundation"]

In [3]:
# Re-aplicar clasificación con el diccionario v1.2

def classify_material(activity_name: str) -> dict:
    if pd.isna(activity_name):
        return {
            "Is_Critical_Material": False,
            "Material_Type": "Non-Critical",
            "Lead_Time_Days": None,
            "Buffer_Days": None,
            "Criticality": "None",
            "Material_Notes": None
        }
    
    name_lower = str(activity_name).lower()
    
    for key in PRIORITY_ORDER:
        rule = MATERIAL_RULES[key]
        for kw in rule["keywords"]:
            if kw in name_lower:
                return {
                    "Is_Critical_Material": True,
                    "Material_Type": rule["material_type"],
                    "Lead_Time_Days": rule["lead_time_days"],
                    "Buffer_Days": rule["buffer_days"],
                    "Criticality": rule["criticality"],
                    "Material_Notes": rule["notes"]
                }
    
    return {
        "Is_Critical_Material": False,
        "Material_Type": "Non-Critical",
        "Lead_Time_Days": None,
        "Buffer_Days": None,
        "Criticality": "None",
        "Material_Notes": None
    }

# Eliminar TODAS las columnas de clasificación que puedan existir (incluso duplicadas)
cols_to_remove = ["Is_Critical_Material", "Material_Type", "Lead_Time_Days", 
                  "Buffer_Days", "Criticality", "Material_Notes"]

# Forma segura de eliminar aunque haya duplicados
df_master = df_master.loc[:, ~df_master.columns.isin(cols_to_remove)]

#  Volver a aplicar la clasificación con el diccionario v1.2
classification = df_master["Name"].apply(classify_material).apply(pd.Series)

#  Concatenar de forma limpia
df_master = pd.concat([df_master, classification], axis=1)

#  Asegurar tipos correctos
df_master["Is_Critical_Material"] = df_master["Is_Critical_Material"].astype(bool)

#  Resumen limpio
print("=" * 70)
print("RESULTADO DEL ETIQUETADO – v1.2 (limpio)")
print("=" * 70)
print(f"\nTotal actividades              : {len(df_master)}")
print(f"Actividades críticas           : {df_master['Is_Critical_Material'].sum()}")
print(f"% críticas                     : {df_master['Is_Critical_Material'].mean()*100:.1f}%")
print("\nDistribución por Material_Type:")
print(df_master[df_master["Is_Critical_Material"]]["Material_Type"].value_counts().to_string())
print("\nDistribución por Criticality:")
print(df_master[df_master["Is_Critical_Material"]]["Criticality"].value_counts().to_string())

In [4]:
# Vista de las actividades Nacelle y Tower (las más importantes)
print("\n" + "=" * 70)
print("ACTIVIDADES CRÍTICAS CLAVE – Nacelle & Tower")
print("=" * 70)

cols = ["ID", "WBS", "Name", "Material_Type", "Criticality", "Lead_Time_Days", "Buffer_Days"]
mask = df_master["Material_Type"].isin(["Nacelle", "Tower"])
display(df_master.loc[mask, cols].sort_values(["Material_Type", "ID"]))

In [5]:
# ============================================================
# DIAGNÓSTICO – Buscar nombres reales de actividades de turbina
# ============================================================

keywords_diagnostic = [
    "tower", "toren", "mast",
    "nacelle", "gondel", "gondels", "machine", "hub",
    "blade", "pala", "wiek", "wieken", "rotor",
    "transformer", "trafo", "transformator",
    "crane", "kraan", "kranen",
    "turbine", "wtg", "generator", "install"
]

print("=" * 70)
print("ACTIVIDADES QUE CONTIENEN KEYWORDS DE TURBINA / MONTAJE")
print("=" * 70)

mask = df_master["Name"].str.lower().str.contains("|".join(keywords_diagnostic), na=False)
df_diag = df_master.loc[mask, ["ID", "WBS", "Name", "Duration_Plan_Days", "Total_Cost_Plan"]].copy()

print(f"\nEncontradas: {len(df_diag)} actividades\n")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)
print(df_diag.to_string(index=False))

# Sección 2 – Material Readiness Date

In [6]:
# ============================================================
# 2. CÁLCULO DE MATERIAL READINESS DATE
# ============================================================

# Aseguramos que Baseline_Start es datetime
if "Baseline_Start" in df_master.columns:
    df_master["Baseline_Start"] = pd.to_datetime(df_master["Baseline_Start"], errors="coerce")
elif "Baseline Start" in df_master.columns:
    df_master["Baseline_Start"] = pd.to_datetime(df_master["Baseline Start"], errors="coerce")
else:
    # Buscamos posibles nombres alternativos
    print("Columnas disponibles relacionadas con fechas:")
    print([c for c in df_master.columns if "start" in c.lower() or "baseline" in c.lower()])

# Función para calcular Material Readiness
def calculate_material_readiness(row):
    if not row["Is_Critical_Material"]:
        return pd.NaT
    
    if pd.isna(row["Baseline_Start"]) or pd.isna(row["Buffer_Days"]):
        return pd.NaT
    
    return row["Baseline_Start"] - pd.Timedelta(days=int(row["Buffer_Days"]))

df_master["Material_Readiness_Date"] = df_master.apply(calculate_material_readiness, axis=1)

# También calculamos los días de anticipación planificada (siempre = Buffer_Days para las críticas)
df_master["Planned_Anticipation_Days"] = df_master.apply(
    lambda r: r["Buffer_Days"] if r["Is_Critical_Material"] else None, axis=1
)

# Resumen de control
print("=" * 70)
print("MATERIAL READINESS DATE – RESUMEN")
print("=" * 70)

crit = df_master[df_master["Is_Critical_Material"]].copy()
print(f"\nActividades críticas con Material Readiness calculada: {crit['Material_Readiness_Date'].notna().sum()}")
print(f"Actividades críticas sin fecha de inicio baseline   : {crit['Material_Readiness_Date'].isna().sum()}")

print("\nEjemplo de las más importantes:")
cols_show = ["ID", "Name", "Material_Type", "Baseline_Start", 
             "Buffer_Days", "Material_Readiness_Date"]
display(crit[cols_show].sort_values("Material_Readiness_Date").head(12))

In [7]:
# Vista específica de Nacelle y las Towers principales
print("\n" + "=" * 70)
print("MATERIAL READINESS – Nacelle & Tower (detalle)")
print("=" * 70)

mask = df_master["Material_Type"].isin(["Nacelle", "Tower"])
display(df_master.loc[mask, ["ID", "Name", "Material_Type", "Baseline_Start", 
                             "Buffer_Days", "Material_Readiness_Date"]]
        .sort_values(["Material_Type", "Baseline_Start"]))

# Sección 3 – Simulación de Riesgos de Suministro + Alarmas de Material


En un dataset histórico real no tenemos el estado real de los pedidos de proveedor. Por tanto, hacemos una simulación realista y defendible basada en:

- La criticidad del material

- La posición temporal de la actividad (cuánto de cerca está del montaje)

- Una desviación simulada de suministro coherente con el comportamiento real del proyecto (el proyecto tuvo problemas de plazo y coste)

In [9]:
# ============================================================
# 3. SIMULACIÓN DE RIESGOS DE SUMINISTRO + ALARMAS
# ============================================================

import numpy as np
np.random.seed(42)   # Reproducibilidad (importante para portfolio)

def simulate_supply_risk(row):
    """
    Simula el riesgo de suministro de forma realista y defendible.
    """
    if not row["Is_Critical_Material"]:
        return pd.Series({
            "Supply_Delay_Days": 0,
            "Material_Status": "Non-Critical",
            "Supply_Risk_Score": 0,
            "Material_Alarm": "OK"
        })
    
    # Base de retraso según criticidad
    base_delay = {
        "Critical": 18,
        "High": 12,
        "Medium-High": 8,
        "Medium": 5
    }.get(row["Criticality"], 5)
    
    # Factor de posición temporal (actividades más tardías tienen más riesgo acumulado)
    # Usamos el mes del Baseline_Start como proxy
    if pd.notna(row["Baseline_Start"]):
        month = row["Baseline_Start"].month
        time_factor = 1 + (month - 3) * 0.08   # más riesgo hacia el final del proyecto
    else:
        time_factor = 1.0
    
    # Ruido controlado
    noise = np.random.normal(0, 6)
    
    supply_delay = max(0, round(base_delay * time_factor + noise))
    
    # Material Status
    buffer = row["Buffer_Days"] if pd.notna(row["Buffer_Days"]) else 14
    
    if supply_delay == 0:
        status = "On Track"
        risk_score = 10
        alarm = "OK"
    elif supply_delay <= buffer * 0.5:
        status = "At Risk"
        risk_score = 35 + supply_delay
        alarm = "WARNING"
    elif supply_delay <= buffer:
        status = "Delayed"
        risk_score = 60 + supply_delay
        alarm = "WARNING"
    else:
        status = "Critical Delay"
        risk_score = min(100, 80 + supply_delay)
        alarm = "CRITICAL"
    
    return pd.Series({
        "Supply_Delay_Days": supply_delay,
        "Material_Status": status,
        "Supply_Risk_Score": int(risk_score),
        "Material_Alarm": alarm
    })

# Aplicar simulación
risk_df = df_master.apply(simulate_supply_risk, axis=1)
df_master = pd.concat([df_master, risk_df], axis=1)

# Resumen de alarmas de material
print("=" * 70)
print("RESUMEN DE RIESGOS Y ALARMAS DE MATERIAL")
print("=" * 70)

crit = df_master[df_master["Is_Critical_Material"]]
print(f"\nActividades críticas analizadas : {len(crit)}")
print("\nDistribución por Material_Status:")
print(crit["Material_Status"].value_counts().to_string())
print("\nDistribución por Material_Alarm:")
print(crit["Material_Alarm"].value_counts().to_string())
print(f"\nSupply Risk Score medio         : {crit['Supply_Risk_Score'].mean():.1f}")
print(f"Máximo Supply Risk Score        : {crit['Supply_Risk_Score'].max()}")

In [10]:
# Vista de las alarmas más relevantes (Nacelle + Towers con mayor riesgo)
print("\n" + "=" * 70)
print("ALARMAS DE MATERIAL – Nacelle & Towers de mayor riesgo")
print("=" * 70)

cols = ["ID", "Name", "Material_Type", "Material_Readiness_Date", 
        "Supply_Delay_Days", "Material_Status", "Supply_Risk_Score", "Material_Alarm"]

mask = (df_master["Is_Critical_Material"]) & (df_master["Material_Type"].isin(["Nacelle", "Tower"]))
display(df_master.loc[mask, cols]
        .sort_values(["Material_Alarm", "Supply_Risk_Score"], ascending=[True, False])
        .head(15))

# Sección 4 – KPIs de Control de Materiales + Guardado de Outputs

In [12]:
# ============================================================
# 4. KPIs DE CONTROL DE MATERIALES
# ============================================================

crit = df_master[df_master["Is_Critical_Material"]].copy()

kpis = {
    "Total_Critical_Materials": len(crit),
    "Pct_On_Track": round((crit["Material_Status"] == "On Track").mean() * 100, 1),
    "Pct_At_Risk": round((crit["Material_Status"] == "At Risk").mean() * 100, 1),
    "Pct_Delayed": round((crit["Material_Status"] == "Delayed").mean() * 100, 1),
    "Pct_Critical_Delay": round((crit["Material_Status"] == "Critical Delay").mean() * 100, 1),
    "Num_CRITICAL_Alarms": int((crit["Material_Alarm"] == "CRITICAL").sum()),
    "Num_WARNING_Alarms": int((crit["Material_Alarm"] == "WARNING").sum()),
    "Avg_Supply_Risk_Score": round(crit["Supply_Risk_Score"].mean(), 1),
    "Max_Supply_Risk_Score": int(crit["Supply_Risk_Score"].max()),
    "Avg_Buffer_Days": round(crit["Buffer_Days"].mean(), 1),
    "Nacelle_Risk_Score": int(crit.loc[crit["Material_Type"] == "Nacelle", "Supply_Risk_Score"].iloc[0]) 
                         if (crit["Material_Type"] == "Nacelle").any() else None
}

print("=" * 70)
print("KPIs DE CONTROL DE MATERIALES – RESUMEN EJECUTIVO")
print("=" * 70)
for k, v in kpis.items():
    print(f"{k:<30}: {v}")

In [13]:
# ============================================================
# 5. GUARDADO DE OUTPUTS
# ============================================================

# 1. Activity Master enriquecido con materiales
output_master = OUTPUT_DIR / "Activity_Master_with_Materials.xlsx"
df_master.to_excel(output_master, index=False)
print(f"\nGuardado: {output_master}")

# 2. Tabla lista para dashboard / Power BI (solo críticas + columnas clave)
dashboard_cols = [
    "ID", "WBS", "Name", "Material_Type", "Criticality",
    "Baseline_Start", "Buffer_Days", "Material_Readiness_Date",
    "Supply_Delay_Days", "Material_Status", "Supply_Risk_Score", "Material_Alarm",
    "Lead_Time_Days", "Material_Notes"
]

df_dashboard = df_master[df_master["Is_Critical_Material"]][dashboard_cols].copy()
df_dashboard = df_dashboard.sort_values(["Material_Alarm", "Supply_Risk_Score"], ascending=[True, False])

output_dashboard = OUTPUT_DIR / "Material_Control_Dashboard_Data.xlsx"
df_dashboard.to_excel(output_dashboard, index=False)
print(f"Guardado: {output_dashboard}")

# 3. Tabla de alarmas (solo WARNING + CRITICAL)
df_alarms = df_dashboard[df_dashboard["Material_Alarm"].isin(["WARNING", "CRITICAL"])].copy()
output_alarms = OUTPUT_DIR / "Material_Alarms.xlsx"
df_alarms.to_excel(output_alarms, index=False)
print(f"Guardado: {output_alarms}")

print("\n" + "=" * 70)
print("NOTEBOOK 05 – OUTPUTS GENERADOS CORRECTAMENTE")
print("=" * 70)